In [ ]:
import numpy as np
import sys
sys.path.append('scripts')

In [76]:
import pdb_to_graph
import mldft_surrogate
import verify_twin_pipeline
import pdb_voxelizier
import cnn_mlp_encoder
import jw_quantum_mapper


In [77]:
protein = "proteins/1I3V.pdb"

In [70]:
# Run track A (CNN)
tensor = pdb_voxelizier.pdb_to_tensor(protein, grid_size=32)
cnn_coefficients = cnn_mlp_encoder.get_hamiltonian(tensor)

In [73]:
print(cnn_coefficients)

[ 0.06939631  0.02254555  0.05586632  0.08423668  0.00764178 -0.01692554
  0.07420833  0.00078711 -0.04123827 -0.01649907]


In [71]:
# 1. Run Track B (ML-DFT)
graph_data = pdb_to_graph.pdb_to_graph(protein, distance_threshold=5.0)
mldft_coefficients = mldft_surrogate.get_mldft_hamiltonian(graph_data, num_qubits=4)


In [74]:
print(mldft_coefficients)

[-0.12033843  0.02013955  0.24219108 -0.03367524  0.3128485   0.09334975
 -0.3436233   0.3207118  -0.13101056 -0.31229836]


In [72]:
# 2. Run the Verification (Assuming you have 'cnn_coefficients' from Track A)
# Use dummy data here just to test the script if needed:
#cnn_coefficients = mldft_coefficients + np.random.normal(0, 0.01, 10)
verify_twin_pipeline.cross_verify_pipelines(cnn_coefficients, mldft_coefficients, num_sites=4)


=== TWIN PIPELINE VERIFICATION REPORT ===

[Checkpoint 1] Coefficient Mean Absolute Error (MAE): 0.203519 eV
-> Status: WARNING (Check spatial mapping drift)

[Checkpoint 2] Physical Ground State Energy (E0)
Track A (3D CNN) E0 : -0.023815 eV
Track B (ML-DFT) E0 : -0.544597 eV
Delta E (Error)     : 0.520781 eV
-> Status: FAIL (Exceeds Chemical Accuracy. Do not send to QPU.)
